In [ ]:
# IMPORTA BIBLIOTECAS
import pandas as pd
import numpy as np
from sklearn.cluster import KMeans
from sklearn.metrics import accuracy_score
from scipy.optimize import linear_sum_assignment
import seaborn as sb

In [ ]:
# CARREGAR O DATASET
df = pd.read_csv('pokemon_gen1.csv')

In [ ]:
# VISUALIZAR O DATAFRAME - MOSTRAR AS PRIMEIRAS LINHAS
df

In [ ]:
# TRANSFORM RÓTULOS DE TEXTO PARA NÚMEROS
from sklearn.preprocessing import LabelEncoder
label_encoder = LabelEncoder()

# USAR LENDÁRIO COMO RÓTULO DE COMPARAÇÃO
# 'Sim' = 1, 'Não' = 0
y_true = label_encoder.fit_transform(df['Lendario'])
print('Classes:', list(label_encoder.classes_))
print('Distribuição:', pd.Series(y_true).value_counts().to_dict())

In [ ]:
# VISUALIZAR A TRANSFORMAÇÃO DE TEXTO PARA NÚMEROS
y_true

In [ ]:
# VISUALIZAR O DATAFRAME ORIGINAL COM LENDÁRIO
sb.pairplot(df, hue='Lendario')

In [ ]:
# USAR APENAS OS STATS DE BATALHA COMO FEATURES
colunas_stats = ['HP', 'Ataque', 'Defesa', 'Ataque_Especial', 'Defesa_Especial', 'Velocidade']
X = df[colunas_stats].values
X

In [ ]:
# NORMALIZAR OS DADOS — padroniza a escala de todas as colunas
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
X = scaler.fit_transform(X)

In [ ]:
# DEFININDO K=2 — Lendário vs Não Lendário
k = 2
print(f'Número de clusters (k): {k}')

# CRIANDO O OBJETO KMEANS PASSANDO O NUMERO DE CLUSTERS
# TREINAMENTO INICIAL DOS DADOS SEM O RÓTULO
kmeans = KMeans(n_clusters=k, random_state=0, n_init=10)
clusters = kmeans.fit_predict(X)
print(clusters)

In [ ]:
# RECONSTRUIU O OBJETO COM NÚMERO DE CLUSTERS E FORÇANDO SEMPRE A COMEÇAR DO MESMO PONTO
kmeans = KMeans(n_clusters=k, random_state=0, n_init=10)

In [ ]:
# TREINAMENTO DO DATAFRAME SEM O RÓTULO UTILIZANDO KMEANS
kmeans.fit(X)

In [ ]:
# EXIBIR OS RÓTULOS QUE FORAM GERADOS
kmeans.labels_

In [ ]:
# CRIANDO UMA NOVA COLUNA NO DATAFRAME ORIGINAL INSERINDO OS LABELS DO DATAFRAME X QUE FOI TREINADO
df['k_classes'] = kmeans.labels_

In [ ]:
# EXIBE AS PRIMEIRAS LINHAS DO DATAFRAME ORIGINAL COM A NOVA COLUNA
df.head(151)

In [ ]:
sb.pairplot(df, hue='k_classes')
sb.pairplot(df, hue='Lendario')

In [ ]:
# IMPORTAÇÃO DA MATRIZ DE CONFUSÃO
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
conf_matrix = confusion_matrix(y_true, clusters)
row_ind, col_ind = linear_sum_assignment(-conf_matrix)

In [ ]:
clusters_mapeados = np.zeros_like(clusters)
for i, j in zip(row_ind, col_ind):
  clusters_mapeados[clusters == i] = j

In [ ]:
# REALIZAR A ACURACIA PARA VERIFICAR A QUANTIDADE DE ACERTOS EM PERCENTUAL
acuracia = accuracy_score(y_true, clusters_mapeados)
print(f"Acurácia: {acuracia * 100:.2f}%")

In [ ]:
import matplotlib.pyplot as plt

disp = ConfusionMatrixDisplay(confusion_matrix=conf_matrix)
disp.plot(cmap='Reds')

plt.title("Matriz de Confusão — Lendário vs Não Lendário")
plt.show()

In [ ]:
# ENCONTRAR O MELHOR K — MÉTODO DO COTOVELO
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans

inercias = []

for k in range(1, 15):
  kmeans = KMeans(n_clusters=k, random_state=0, n_init=10)
  kmeans.fit(X)
  inercias.append(kmeans.inertia_)

plt.plot(range(1, 15), inercias, marker='o')
plt.xlabel('Num de clusters')
plt.ylabel('Inércias')
plt.title('Método Cotovelo')
plt.show()
print(inercias)

In [ ]:
from sklearn.metrics import silhouette_score

silhouette_scores = []

for k in range(2, 15):
  kmeans = KMeans(n_clusters=k, random_state=0, n_init=10)
  labels = kmeans.fit_predict(X)
  score = silhouette_score(X, labels)
  silhouette_scores.append(score)

plt.plot(range(2, 15), silhouette_scores, marker='o')
plt.xlabel('Num de clusters')
plt.ylabel('Índice de Silhueta')
plt.title('Índice de Silhueta por K')
plt.show()
print(silhouette_scores)